___
# <center>Aula 5, completa: Gramática de Gráficos</center>
___

## Aula 05

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * descrever qualquer gráfico como dados, estética e geometria;
 * escolher a geometria a partir do tipo da variável;
 * fazer gráficos de uma variável só, categórica ou numérica, no plotnine;
 * explicar a diferença entre pintar de uma cor e mapear uma variável em cor;
 * repartir um gráfico em facetas.

Esta é a **versão completa** da aula 5: a gramática de gráficos do começo ao
fim, com o que foi projetado e também o que não coube no telão (histograma,
boxplot, rótulos, facetas e três exercícios).

> Um gráfico estatístico é um **mapeamento de variáveis (colunas) em aspectos
> estéticos de formas geométricas**.

A ideia é a mesma do encadeamento. Lá, uma análise era uma sequência de operações
somadas com `.`. Aqui, um gráfico é uma sequência de camadas somadas com `+`.

> 📌 Se você quer **acompanhar a aula** rodando o mesmo código que está no
> telão, use o notebook **aula05**, que segue os slides na ordem e traz a
> gincana. Este aqui é para depois: é a referência do **Projeto 02**, na
> quinta-feira.


___
<div id="indice"></div>

## Índice

- [A tabela de hoje](#dados)

- [Encadear operações](#encadear)
    - [Do jeito da aula 3](#antigo)
    - [🔗 A mesma coisa, encadeada](#encadeado)
    - [Os cinco verbos](#verbos)
    - [A ordem importa](#ordem)

- [O que é um gráfico](#definicao)

- [Os três elementos obrigatórios](#gramatica)
    - [1️⃣ Os dados](#elem-dados)
    - [2️⃣ A estética: aes()](#elem-aes)
    - [3️⃣ A geometria: geom_](#elem-geom)

- [Uma variável categórica: barras](#categorica)
    - [Barras deitadas](#flip)
    - [🎨 Pintar não é mapear](#fill)

- [Uma variável numérica: histograma](#numerica)
    - [O número de caixas muda a leitura](#bins)
    - [O boxplot](#outras-geoms)

- [Rótulos: labs()](#labs)

- [Facetas: facet_wrap()](#facetas)

- [Que gráfico usar para cada variável](#escolha)

- [Exercícios](#exercicios)
    - [EXERCÍCIO 1: barras de comarca](#ex1)
    - [EXERCÍCIO 2: histograma da ementa](#ex2)
    - [EXERCÍCIO 3: o gráfico da sua pergunta](#ex3)

- [RESUMO](#resumo)


___
<div id="dados"></div>

# A tabela de hoje

A mesma base de apelações criminais da aula 4, e a mesma tabela `penas` que
estava impressa na folha da dinâmica.


O plotnine não vem instalado no Google Colab. Tire o `#` da segunda linha, rode
uma vez, e ponha o `#` de volta:


In [ ]:
# no Colab, rode uma vez:
# %pip install -q plotnine


In [ ]:
import pandas as pd
from plotnine import *

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


> 🤔 `from plotnine import *` traz todos os nomes da biblioteca de uma vez:
> `ggplot`, `aes`, `geom_bar` e companhia. Fora do plotnine essa forma é
> desaconselhada, porque você não sabe mais de onde cada nome veio. Aqui ela é a
> convenção, porque um gráfico junta cinco ou seis desses nomes numa expressão só.


In [ ]:
criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")

criminal["regime"] = pd.Categorical(
    criminal["regime_inicial"],
    categories=["aberto", "semiaberto", "fechado"],
    ordered=True,
)

penas = (
    criminal
    .dropna(subset=["regime", "pena_anos"])
    .query("pena_anos <= 30")
)

criminal.shape, penas.shape


O corte em 30 anos tira as penas implausíveis, aquelas em que a leitura da ementa
pegou o número errado. É uma decisão de análise: está escrita no código, e
qualquer pessoa que leia o notebook sabe que ela existe.

Antes de qualquer coisa, olhe as duas tabelas. `.head()` mostra as primeiras
linhas, e é a primeira coisa a fazer depois de ler um arquivo: se a tabela veio
torta, você descobre agora e não daqui a vinte células.


In [ ]:
criminal.head()


`penas` é a mesma tabela com duas decisões já tomadas: fora quem não tem regime
ou pena, e fora as penas acima de 30 anos. Repare que ela tem uma coluna a mais,
`regime`, que é a versão **ordenada** de `regime_inicial`.


In [ ]:
penas.head()


[Volta ao Índice](#indice)


___
<div id="encadear"></div>

# Encadear operações

Esta seção é o espelho dos slides, para você acompanhar rodando. Nada aqui é
novo em relação ao que está sendo projetado.

A pergunta é a mesma da aula 4:

> Nas apelações criminais do TJSP, a proporção de acórdãos que mencionam
> reincidência varia conforme o regime inicial fixado?


<div id="antigo"></div>

### Do jeito da aula 3

Uma variável nova a cada operação. Funciona, e responde a pergunta:


In [ ]:
apelacoes = criminal[criminal["classe"] == "Apelação Criminal"]
com_regime = apelacoes.dropna(subset=["regime_inicial"])
fechado = com_regime[com_regime["regime_inicial"] == "fechado"]
semiaberto = com_regime[com_regime["regime_inicial"] == "semiaberto"]
aberto = com_regime[com_regime["regime_inicial"] == "aberto"]

pd.Series({
    "fechado": fechado["houve_reincidencia"].mean(),
    "semiaberto": semiaberto["houve_reincidencia"].mean(),
    "aberto": aberto["houve_reincidencia"].mean(),
}).round(3)


Três problemas: **seis variáveis** que existem só para chegar num resultado,
**nomes que não dizem nada** (`com_regime` vai ser reaproveitado por engano daqui
a três células) e **não escala** (um quarto regime obriga a escrever mais uma
linha e a lembrar de incluí-la).


<div id="encadeado"></div>

### 🔗 A mesma coisa, encadeada

Nenhuma variável intermediária, e a ordem das operações é a ordem das linhas.


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["regime_inicial"])
    .groupby("regime_inicial")
    .agg(proporcao=("houve_reincidencia", "mean"))
    .round(3)
)


Leia de cima para baixo, como uma frase: pegue `criminal`, fique só com as
apelações, descarte quem não tem regime, faça as pilhas por regime, e calcule a
proporção de cada pilha.

**Por que os parênteses.** Dentro de um par de parênteses o python deixa você
quebrar a linha à vontade. Sem eles, `criminal` seguido de quebra de linha e
`.query(...)` é erro de sintaxe. Eles existem só para você pôr uma operação por
linha:

```python
resultado = (
    tabela
    .operacao_1(...)
    .operacao_2(...)
)
```


<div id="verbos"></div>

### Os cinco verbos

Cinco operações resolvem quase toda análise descritiva. São as mesmas que estão
nas cartas da gincana.


**1. `.query()` escolhe linhas.** A condição vai escrita como texto, e o valor
comparado vai entre aspas simples. Colunas de verdadeiro e falso dispensam a
comparação: basta o nome.


In [ ]:
criminal.query("eh_trafico").shape


**2. `[[...]]` escolhe colunas.** São **dois** pares de colchetes. Um só devolve
a coluna solta, e o encadeamento acaba ali.


In [ ]:
(
    criminal
    [["processo", "comarca", "regime_inicial", "pena_anos"]]
    .head(3)
)


**3. `.sort_values()` ordena.** `ascending=False` põe o maior primeiro.


In [ ]:
(
    criminal
    .sort_values("pena_anos", ascending=False)
    [["processo", "assunto", "pena_anos"]]
    .head(5)
)


> ⚠️ Olhe o resultado acima com atenção. As maiores penas da base estão num
> furto, num estelionato e numa apropriação indébita, e chegam a 75 anos. A pena
> foi lida pegando o primeiro número seguido de "anos" na ementa, e às vezes o
> número está errado. É por isso que a tabela `penas` corta em 30.

**4. `.groupby()` e `.agg()` agregam por grupo.** O `groupby` faz as pilhas e o
`agg` calcula uma estatística em cada uma, devolvendo **uma linha por grupo**.
Depois dele, a coluna de agrupamento vira índice, e o `.reset_index()` traz ela
de volta para dentro da tabela.


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(
        n=("processo", "size"),
        pena_mediana=("pena_anos", "median"),
        prop_reincidencia=("houve_reincidencia", "mean"),
    )
    .reset_index()
    .round(3)
)


A média de uma coluna de verdadeiro e falso é a **proporção** de verdadeiros. Foi
assim que saiu `prop_reincidencia`.

**5. `.assign()` cria uma coluna nova**, dentro do encadeamento. O nome novo vai
à esquerda do `=`. A coluna criada só existe da linha seguinte em diante, então
o `.assign()` precisa vir **antes** de qualquer operação que use ela.


In [ ]:
(
    criminal
    .assign(eh_capital=criminal["comarca"] == "São Paulo")
    .groupby("eh_capital")
    .agg(n=("processo", "size"))
    .reset_index()
)


E duas operações de apoio, que não são verbos novos:

* `.dropna(subset=["coluna"])` descarta as linhas em que **aquela** coluna está
  vazia. Sem o `subset`, descarta toda linha que tenha qualquer campo vazio, o
  que quase sempre é mais do que você queria.
* `.head(n)` fica com as `n` primeiras linhas **da tabela como ela está naquele
  ponto**. Guarde esta frase: a próxima seção é sobre ela.


<div id="ordem"></div>

### A ordem importa

As duas células abaixo têm exatamente as mesmas operações. Só trocam duas linhas
de lugar.


Primeiro do jeito certo: ordenar e **depois** cortar.


In [ ]:
(
    criminal
    .sort_values("pena_anos", ascending=False)
    .head(5)
    [["processo", "pena_anos"]]
)


Agora ao contrário: cortar e **depois** ordenar.


In [ ]:
(
    criminal
    .head(5)
    .sort_values("pena_anos", ascending=False)
    [["processo", "pena_anos"]]
)


São cinco acórdãos quaisquer, os cinco primeiros da tabela, ordenados entre si.
Nada a ver com as maiores penas.

O `.head(5)` não sabe o que você queria. Ele pega as cinco primeiras linhas da
tabela **como ela está naquele ponto**, e é por isso que quase sempre vem por
último.


[Volta ao Índice](#indice)


___
<div id="definicao"></div>

# O que é um gráfico

Antes de desenhar qualquer coisa, uma definição:

> Um gráfico estatístico é um **mapeamento de variáveis (colunas)** em
> **aspectos estéticos** de **formas geométricas**.

Quatro expressões fazem o trabalho:

| a expressão | o que quer dizer |
|---|---|
| mapeamento | uma ligação: cada valor da coluna vira um valor visual |
| variáveis (colunas) | o que sai da tabela, e nada mais |
| aspectos estéticos | posição, altura, cor, tamanho, forma |
| formas geométricas | barra, ponto, linha, caixa |

Repare no que a definição **não** diz: nada sobre que biblioteca usar, que cor
fica bonita ou que tipo de gráfico escolher. Ela diz o que precisa ser
**decidido**, e é essa lista de decisões que a próxima seção transforma em
código.


[Volta ao Índice](#indice)


___
<div id="gramatica"></div>

# Os três elementos obrigatórios

Todo gráfico do plotnine é uma soma de camadas, escrita dentro de parênteses,
uma por linha. As três primeiras são obrigatórias:

```python
(
    ggplot(tabela)          # 1. os dados
    + aes(x="coluna")       # 2. a estética: que variável vai em que lugar
    + geom_barra()          # 3. a geometria: que desenho aparece na tela
)
```

Repare no formato: é o mesmo da aula 4, com `+` no lugar de `.`. Abre parêntese,
uma camada por linha, e o parêntese permite quebrar a linha.


<div id="elem-dados"></div>

### 1️⃣ Os dados

`ggplot(penas)` diz de qual tabela o gráfico sai. Só isso já é um gráfico
válido, e é um retângulo vazio: ainda não dissemos o que desenhar.


In [ ]:
ggplot(penas)


<div id="elem-aes"></div>

### 2️⃣ A estética: aes()

**Estética** é a ligação entre uma coluna da tabela e uma propriedade visual do
gráfico. As mais usadas são `x`, `y`, `fill` (preenchimento), `color` (traço) e
`size`.

`aes(x="regime")` diz: a coluna `regime` vai no eixo horizontal. O plotnine
agora sabe do que se trata o eixo, e já desenha a escala.


In [ ]:
(
    ggplot(penas)
    + aes(x="regime")
)


<div id="elem-geom"></div>

### 3️⃣ A geometria: geom_

**Geometria** é o desenho: barra, coluna, ponto, linha, caixa. É a camada que
finalmente põe tinta no papel.


`geom_bar()` conta quantas linhas existem em cada categoria e desenha uma barra
com a altura dessa contagem. Você não precisa contar antes: a contagem é parte
da geometria.

✔️ **Uso do `geom_bar()`**

```python
# Sintaxe geral:
(
    ggplot(tabela)
    + aes(x="coluna_categorica")
    + geom_bar()
)
```

Documentação oficial: [geom_bar()](https://plotnine.org/reference/geom_bar.html)


In [ ]:
(
    ggplot(penas)
    + aes(x="regime")
    + geom_bar()
)


As barras saem na ordem aberto, semiaberto, fechado porque `regime` é uma
categórica **ordenada**, declarada lá em cima. Se fosse texto comum, o plotnine
usaria a ordem alfabética, e o gráfico diria que semiaberto vem depois de
fechado, o que não é verdade em nada.

É a aula 2 cobrando a fatura: declarar o tipo certo não é preciosismo, é o que
faz o gráfico sair certo.


[Volta ao Índice](#indice)


___
<div id="categorica"></div>

# Uma variável categórica: barras

Com uma variável categórica, a pergunta quase sempre é *quantos casos em cada
categoria*, e a resposta quase sempre é uma barra.


**✍️ Agora você.** Troque a variável do eixo x e faça as barras da classe processual.


In [ ]:
(
    ggplot(penas)
    + aes(x="________")
    + geom_bar()
)


Os rótulos ficaram um por cima do outro. Isso acontece sempre que a categórica
tem nomes longos, e a solução é deitar as barras.


<div id="flip"></div>

### Barras deitadas

`coord_flip()` troca os eixos depois que o gráfico já está montado. Nada muda nos
dados nem na geometria: é só o sistema de coordenadas.


In [ ]:
(
    ggplot(penas)
    + aes(x="classe")
    + geom_bar()
    + coord_flip()
)


<div id="fill"></div>

### 🎨 Pintar não é mapear

Esta é a distinção que o desafio da folha pedia, e é a ideia mais importante da
aula.

`fill` **fora** do `aes()` é uma escolha de tinta. Vale para tudo, não representa
nada, e não gera legenda:


In [ ]:
(
    ggplot(penas)
    + aes(x="regime")
    + geom_bar(fill="#E50505")
)


`fill` **dentro** do `aes()` é um mapeamento. A cor passa a representar uma
variável, cada barra se reparte, e aparece uma legenda, porque agora a cor
significa alguma coisa:


In [ ]:
(
    ggplot(penas)
    + aes(x="regime", fill="houve_reincidencia")
    + geom_bar()
)


> ⚠️ A regra vale para todas as estéticas, não só para `fill`. Dentro do `aes()`
> o argumento recebe o **nome de uma coluna**. Fora do `aes()` ele recebe um
> **valor fixo**. Trocar os dois de lugar é o erro mais comum de quem está
> começando, e o sintoma é sempre o mesmo: apareceu uma legenda que você não
> queria, ou sumiu a legenda que você queria.

Este gráfico já responde à pergunta da aula 4, agora sem tabela: a fatia colorida
cresce conforme o regime fica mais severo. Guarde a pergunta que ele **não**
responde: as três barras têm tamanhos diferentes, então comparar as fatias de
olho é comparar proporções em bases diferentes. Voltamos a isso na aula 6.


**✍️ Agora você.** Faça as barras de `regime` repartidas por `eh_trafico`.


In [ ]:
(
    ggplot(penas)
    + aes(x="regime", ________="eh_trafico")
    + geom_bar()
)


[Volta ao Índice](#indice)


___
<div id="numerica"></div>

# Uma variável numérica: histograma

Com uma variável numérica contínua não dá para contar por valor: quase toda pena
aparece uma ou duas vezes. O histograma resolve isso cortando o eixo em faixas de
mesma largura, as **caixas**, e contando quantos casos caem em cada uma.


`bins` é o número de caixas. Sem ele, o plotnine escolhe um número e avisa que
escolheu.

✔️ **Uso do `geom_histogram()`**

```python
# Sintaxe geral:
(
    ggplot(tabela)
    + aes(x="coluna_numerica")
    + geom_histogram(bins=10)
)
```

Documentação oficial: [geom_histogram()](https://plotnine.org/reference/geom_histogram.html)


In [ ]:
(
    ggplot(penas)
    + aes(x="pena_anos")
    + geom_histogram(bins=10)
)


A leitura: a maior parte das penas está abaixo de cinco anos, e a distribuição
tem uma cauda longa à direita. É a mesma assimetria que separava a média da
mediana na aula 3, agora visível de uma vez.


<div id="bins"></div>

### O número de caixas muda a leitura

São as cartas 5 e 6 da dinâmica. Mesmos dados, mesma geometria, duas histórias:


In [ ]:
(
    ggplot(penas)
    + aes(x="pena_anos")
    + geom_histogram(bins=40)
)


Com 40 caixas aparecem picos e buracos que são, em boa parte, o tamanho da
amostra, e não um padrão das penas. Com 5 caixas aconteceria o contrário: a
cauda sumiria dentro de uma barra só.

Qual está certo? Nenhum. **O número de caixas é decisão de quem analisa**, e é
por isso que ele fica escrito no código, à vista. Um bom hábito é olhar dois ou
três valores antes de escolher.


**✍️ Agora você.** Rode o mesmo histograma com 5 caixas e compare com os dois de cima.


In [ ]:
(
    ggplot(penas)
    + aes(x="pena_anos")
    + geom_histogram(bins=________)
)


<div id="outras-geoms"></div>

### O boxplot

Trocar a geometria é trocar uma linha, e a mesma variável numérica aceita mais
de uma forma. Depois do histograma, a outra que vale conhecer é o boxplot:


O boxplot é o resumo da aula 3 virado desenho: a caixa vai do primeiro ao
terceiro quartil, a linha do meio é a mediana, e os pontos soltos são os valores
distantes do resto.

Ele precisa de duas estéticas, uma para o eixo do grupo e outra para o valor.
Como aqui não há grupo nenhum, o `x` recebe um texto fixo, entre aspas duplas e
simples, só para o plotnine ter o que pôr no eixo:


In [ ]:
(
    ggplot(penas)
    + aes(x='"todos os acórdãos"', y="pena_anos")
    + geom_boxplot()
)


> 🤔 Um boxplot de uma variável só serve para pouca coisa. O boxplot fica útil
> quando há um grupo no eixo `x`, e aí ele compara várias distribuições lado a
> lado. Isso é a aula 6.


[Volta ao Índice](#indice)


___
<div id="labs"></div>

# Rótulos: labs()

Por padrão, os eixos recebem o nome da coluna. `pena_anos` e `count` servem para
você, e não servem para ninguém mais. `labs()` troca os rótulos, e é a diferença
entre um gráfico de rascunho e um gráfico que pode ir para o relatório.


✔️ **Uso do `labs()`**

```python
# Sintaxe geral:
labs(title="Título", x="Eixo x", y="Eixo y", fill="Legenda")
```

Documentação oficial: [labs()](https://plotnine.org/reference/labs.html)


In [ ]:
(
    ggplot(penas)
    + aes(x="pena_anos")
    + geom_histogram(bins=10, fill="#E50505")
    + labs(
        title="Penas fixadas em apelações criminais do TJSP",
        subtitle="Acórdãos com pena identificada na ementa, até 30 anos",
        x="Pena (anos)",
        y="Acórdãos",
    )
    + theme_minimal()
)


`theme_minimal()` é uma decisão só de aparência: tira o fundo cinza e deixa o
gráfico mais limpo para impressão. Existem outros temas prontos, e nenhum deles
muda os dados.


**✍️ Agora você.** Ponha título e rótulos no gráfico de barras do regime.


In [ ]:
(
    ggplot(penas)
    + aes(x="regime")
    + geom_bar(fill="#E50505")
    + ________(
        title="Regime inicial fixado",
        x="________",
        y="Acórdãos",
    )
    + theme_minimal()
)


[Volta ao Índice](#indice)


___
<div id="facetas"></div>

# Facetas: facet_wrap()

A faceta não faz um gráfico novo. Ela **repete** o mesmo gráfico, uma vez para
cada valor de uma variável, com os mesmos eixos, para que os painéis sejam
comparáveis.


✔️ **Uso do `facet_wrap()`**

```python
# Sintaxe geral:
facet_wrap("coluna_categorica")
```

Documentação oficial: [facet_wrap()](https://plotnine.org/reference/facet_wrap.html)


In [ ]:
(
    ggplot(penas)
    + aes(x="pena_anos")
    + geom_histogram(bins=10)
    + facet_wrap("regime")
    + labs(x="Pena (anos)", y="Acórdãos")
    + theme_minimal()
)


Agora dá para ver o que a tabela da aula 4 dizia com números: a distribuição das
penas se desloca para a direita conforme o regime fica mais severo.

Faceta ou cor? As duas mostram a mesma informação. A cor põe tudo junto e facilita
comparar o total; a faceta separa e facilita comparar o formato de cada grupo.
Com três categorias, qualquer uma serve. Com dez, faceta.


**✍️ Agora você.** Reparta o histograma por `eh_trafico` em vez de por regime.


In [ ]:
(
    ggplot(penas)
    + aes(x="pena_anos")
    + geom_histogram(bins=10)
    + ________("eh_trafico")
    + theme_minimal()
)


[Volta ao Índice](#indice)


___
<div id="escolha"></div>

# Que gráfico usar para cada variável

O tipo da variável, que é a aula 2, decide a geometria. Para uma variável de cada
vez:

| tipo da variável | exemplo na base | geometria | o que você lê |
|---|---|---|---|
| categórica nominal | `comarca`, `classe` | `geom_bar()` + `coord_flip()` | quantos casos em cada categoria |
| categórica ordinal | `regime` | `geom_bar()` | o mesmo, na ordem que importa |
| binária | `houve_reincidencia` | `geom_bar()` | quantos sim e quantos não |
| numérica discreta | `n_palavras_ementa` | `geom_histogram()` | onde os valores se concentram |
| numérica contínua | `pena_anos` | `geom_histogram()`, `geom_boxplot()` | formato, centro e dispersão |

Duas armadilhas frequentes:

* **Histograma de categórica** não existe. Se a variável é texto, é barra.
* **Barras de uma variável contínua** também não: cada valor viraria uma barra de
  altura 1, e o gráfico não diria nada. Se for contínua, é histograma.


[Volta ao Índice](#indice)


___
<div id="exercicios"></div>

# Exercícios


<div id="ex1"></div>

### EXERCÍCIO 1

Faça um gráfico de barras das dez comarcas com mais acórdãos, deitado, com título
e rótulos.

A parte de pandas é a da aula 4: agrupe, conte, ordene e corte. Depois use
`geom_col()`, que desenha barras com a altura que **você** informou em `y`, em
vez de contar as linhas como o `geom_bar()`.


In [ ]:
top_comarcas = (
    criminal
    .groupby("________")
    .agg(n=("processo", "size"))
    .reset_index()
    .sort_values("n", ascending=________)
    .head(________)
)

(
    ggplot(top_comarcas)
    + aes(x="comarca", y="________")
    + geom_col(fill="#E50505")
    + ________()
    + labs(title="Comarcas com mais acórdãos", x="Comarca", y="Acórdãos")
    + theme_minimal()
)


<div id="ex2"></div>

### EXERCÍCIO 2

Faça um histograma do tamanho da ementa, `n_palavras_ementa`, com 20 caixas,
repartido por regime, com título e rótulos.


In [ ]:
(
    ggplot(penas)
    + aes(x="________")
    + geom_histogram(bins=________)
    + ________("regime")
    + labs(
        title="Tamanho da ementa por regime inicial",
        x="Palavras na ementa",
        y="Acórdãos",
    )
    + theme_minimal()
)


<div id="ex3"></div>

### EXERCÍCIO 3

Escreva, em uma frase, uma pergunta descritiva sobre esta base que possa ser
respondida com **uma variável só**. Depois faça o gráfico que a responde, com
título e rótulos, e escreva embaixo, em duas linhas, o que ele mostra e o que ele
não permite concluir.

É o mesmo exercício que vocês vão repetir na pesquisa de campo, com os dados de
vocês.


In [ ]:
# pergunta:

(
    ggplot(________)
    + aes(x="________")
    + ________
    + labs(
        title="________",
        x="________",
        y="________",
    )
    + theme_minimal()
)

# mostra:
# não permite concluir:


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

Um gráfico é uma soma de camadas, escrita dentro de parênteses, uma por linha.
Três são obrigatórias, o resto é ajuste.

| camada | para quê |
|---|---|
| `ggplot(tabela)` | de qual tabela o gráfico sai |
| `+ aes(x=..., fill=...)` | que coluna vai em que propriedade visual |
| `+ geom_bar()` | contar categorias e desenhar barras |
| `+ geom_histogram(bins=n)` | cortar uma variável numérica em caixas e contar |
| `+ geom_boxplot()` | mediana, quartis e valores distantes |
| `+ geom_col()` | barras com a altura que você informou em `y` |
| `+ coord_flip()` | deitar as barras |
| `+ facet_wrap("coluna")` | repetir o mesmo gráfico por categoria |
| `+ labs(title=..., x=..., y=...)` | rótulos que outra pessoa entende |
| `+ theme_minimal()` | aparência |


In [ ]:
#=> DADOS: a tabela, já filtrada e com os tipos declarados
(
    ggplot(penas)

    #=> ESTÉTICA: nome de coluna dentro do aes() é mapeamento
    + aes(x="pena_anos")

    #=> GEOMETRIA: valor fixo fora do aes() é só tinta
    + geom_histogram(bins=10, fill="#E50505")

    #=> FACETA: o mesmo gráfico, repetido por categoria
    + facet_wrap("regime")

    #=> RÓTULOS: sem isso o eixo diz "count"
    + labs(
        title="Penas por regime inicial",
        x="Pena (anos)",
        y="Acórdãos",
    )

    #=> TEMA: só aparência
    + theme_minimal()
)


**Três regras que valem sempre:**

1. Dentro do `aes()` vai **nome de coluna**; fora do `aes()` vai **valor fixo**.
   Legenda que apareceu sem ser chamada é quase sempre isso.
2. O tipo da variável escolhe a geometria: categórica pede barra, numérica pede
   histograma.
3. Todo número que você escolheu, como `bins`, é uma decisão de análise. Ele fica
   no código para que a decisão seja discutível.


[Volta ao Índice](#indice)
